# XSAMSum Evaluation — ROUGE + BERTScore

This notebook evaluates Chinese and English summaries on the XSAMSum test set using the multilingual ROUGE toolkit and BERTScore. Each metric for each language has its own cell so you can re-run any one of them without recomputing the others — useful because BERTScore takes a while to load the encoder and cache embeddings.

**Structure**
1. Setup: imports, config, helpers, load dataset
2. Chinese evaluation — ROUGE, then BERTScore (raw), then combine & save
3. English evaluation — ROUGE, then BERTScore (raw), then BERTScore (rescaled), then combine & save

**Language flag note.** The multilingual ROUGE toolkit expects full language names (`"chinese"`, `"english"`). BERTScore expects ISO codes (`"zh"`, `"en"`). Both are kept in `LANG_CONFIG` — don't mix them up.

## 1. Setup

In [ ]:
import os
import pandas as pd
from datasets import load_dataset
from rouge_score import rouge_scorer
from bert_score import BERTScorer

pd.set_option('display.max_colwidth', 120)

In [ ]:
# Config
DATASET_NAME = "XSAMSum"
BASE_DIR = "sample_data"

# Per-language config: (reference field, predictions file, BERTScore model, BERTScore num_layers,
#                       report_rescaled, rouge_lang)
# - ClidSum uses chinese-bert-wwm-ext for Chinese; num_layers=8 matches bert-base-chinese optimal layer per BERTScore paper.
# - BERTScore paper recommends roberta-large for English; num_layers=None lets bert_score pick the recommended layer.
# - report_rescaled=True adds a rescaled-with-baseline F1 column. bert_score ships baselines for roberta-large
#   but not for chinese-bert-wwm-ext, so this is only enabled for English.
# - rouge_lang is the language NAME expected by the multilingual ROUGE toolkit (full name, not ISO code).
#   BERTScore uses the ISO code (dict key) while ROUGE needs the full name.
LANG_CONFIG = {
    "zh": ("summary_zh", "bart_predictions.txt",     "hfl/chinese-bert-wwm-ext", 8,    False, "chinese"),
    "en": ("summary",    "mbart_predictions_en.txt", "roberta-large",            None, True,  "english"),
}

# ClidSum paper uses R-1 / R-2 / R-L
ROUGE_TYPES = ["rouge1", "rouge2", "rougeL"]
TOP_N_WORST = 10

In [ ]:
# Helper functions
def load_predictions(path):
    """Load predictions from a text file, one summary per line."""
    with open(path, "r", encoding="utf-8") as f:
        return f.read().splitlines()


def compute_rouge(predictions, references, rouge_types, language):
    """Compute per-pair and corpus-level ROUGE F1 scores.

    `language` is the full language name expected by the multilingual ROUGE toolkit,
    e.g. "chinese" or "english" — not an ISO code.
    """
    scorer = rouge_scorer.RougeScorer(
        rouge_types=rouge_types,
        lang=language,
    )

    pair_scores = []
    aggregated = {rt: 0.0 for rt in rouge_types}

    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)  # (reference, hypothesis)
        pair = {rt: round(scores[rt].fmeasure * 100, 2) for rt in rouge_types}
        pair_scores.append(pair)
        for rt in rouge_types:
            aggregated[rt] += scores[rt].fmeasure

    n = len(predictions)
    corpus_scores = {rt: round(aggregated[rt] / n * 100, 2) for rt in rouge_types}
    return corpus_scores, pair_scores


def compute_bertscore(
    predictions, references, model_type, lang,
    num_layers=None, rescale_with_baseline=False,
    batch_size=32, verbose=True,
):
    """Compute per-pair and corpus-level BERTScore F1.

    `lang` here is the ISO code ("zh", "en") that the bert_score library expects.
    """
    predictions = list(predictions)
    references = list(references)
    scorer_kwargs = dict(
        model_type=model_type,
        lang=lang,
        batch_size=batch_size,
        rescale_with_baseline=rescale_with_baseline,
    )
    if num_layers is not None:
        scorer_kwargs["num_layers"] = num_layers
    scorer = BERTScorer(**scorer_kwargs)

    # Fix OverflowError on long inputs
    scorer._tokenizer.model_max_length = 512

    P, R, F1 = scorer.score(
        predictions, references,
        verbose=verbose, batch_size=batch_size,
    )

    pair_scores = [round(F1[i].item() * 100, 2) for i in range(len(predictions))]
    corpus_scores = {"f1": round(F1.mean().item() * 100, 2)}
    return corpus_scores, pair_scores

In [ ]:
# Load the XSAMSum dataset
data_files = {
    "train":      os.path.join(BASE_DIR, "train.json"),
    "validation": os.path.join(BASE_DIR, "val.json"),
    "test":       os.path.join(BASE_DIR, "test.json"),
}
dataset_dict = load_dataset("json", data_files=data_files)
print(dataset_dict)

## 2. Chinese evaluation

### 2a. Load Chinese references & predictions

In [ ]:
ref_field_zh, preds_path_zh, bertscore_model_zh, num_layers_zh, report_rescaled_zh, rouge_lang_zh = LANG_CONFIG["zh"]

references_zh = dataset_dict["test"][ref_field_zh]
predictions_zh = load_predictions(preds_path_zh)
assert len(predictions_zh) == len(references_zh), (
    f"length mismatch: {len(predictions_zh)} preds vs {len(references_zh)} refs"
)
print(f"Loaded {len(predictions_zh)} Chinese prediction/reference pairs")

### 2b. Chinese ROUGE

Multilingual ROUGE toolkit with `lang="chinese"` — this enables Chinese word segmentation (jieba) before n-gram counting. Passing an unrecognized language (e.g. `"zh"`) silently falls back to whitespace tokenization, which for Chinese means character-level matching and substantially inflated scores.

In [ ]:
rouge_corpus_zh, rouge_pairs_zh = compute_rouge(
    predictions_zh, references_zh,
    rouge_types=ROUGE_TYPES, language=rouge_lang_zh,
    use_stemmer=True
)

print("Chinese ROUGE:")
print(f"  ROUGE-1 (R1) : {rouge_corpus_zh['rouge1']:.2f}")
print(f"  ROUGE-2 (R2) : {rouge_corpus_zh['rouge2']:.2f}")
print(f"  ROUGE-L (R-L): {rouge_corpus_zh['rougeL']:.2f}")

### 2c. Chinese BERTScore (raw)

Uses `hfl/chinese-bert-wwm-ext` with `num_layers=8`, matching the ClidSum evaluation protocol. Raw F1 only — the `bert_score` library doesn't ship a precomputed baseline for this model, so rescaling isn't available.

In [ ]:
bs_raw_corpus_zh, bs_raw_pairs_zh = compute_bertscore(
    predictions_zh, references_zh,
    model_type=bertscore_model_zh,
    lang="zh",
    num_layers=num_layers_zh,
    rescale_with_baseline=False,
    batch_size=32,
)
print(f"Chinese BERTScore F1 (raw): {bs_raw_corpus_zh['f1']}")

### 2d. Combine Chinese results, inspect, and save

In [ ]:
# Corpus-level scores
corpus_scores_zh = rouge_corpus_zh | {"bs_f1_raw": bs_raw_corpus_zh["f1"]}
corpus_df_zh = pd.DataFrame(corpus_scores_zh, index=[0])
print(f"Corpus Eval Scores (Chinese) of {DATASET_NAME}")
print(corpus_df_zh)

In [ ]:
# Pair-level scores
pair_cols_zh = (
    {"reference": references_zh, "prediction": predictions_zh}
    | pd.DataFrame(rouge_pairs_zh).to_dict("list")
    | {"bs_f1_raw": bs_raw_pairs_zh}
)
pair_results_df_zh = pd.DataFrame(pair_cols_zh)

print("── BERTScore F1 distribution (Chinese) ──")
print(pair_results_df_zh[["bs_f1_raw"]].describe().round(2))

print("\n── rougeL distribution (Chinese) ──")
print(pair_results_df_zh["rougeL"].describe().round(2))

In [ ]:
worst_zh = pair_results_df_zh.nsmallest(TOP_N_WORST, "rougeL")
print(f"── Top {TOP_N_WORST} worst Chinese examples by ROUGE-L of {DATASET_NAME} ──")
worst_zh

In [ ]:
corpus_df_zh.to_csv(f"corpus_scores_zh_{DATASET_NAME}.csv", index=False, encoding="utf-8-sig")
pair_results_df_zh.to_csv(f"pair_scores_zh_{DATASET_NAME}.csv", index=True, encoding="utf-8-sig")
print("Saved Chinese scores to CSV.")

## 3. English evaluation

### 3a. Load English references & predictions

In [ ]:
ref_field_en, preds_path_en, bertscore_model_en, num_layers_en, report_rescaled_en, rouge_lang_en = LANG_CONFIG["en"]

references_en = dataset_dict["test"][ref_field_en]
predictions_en = load_predictions(preds_path_en)
assert len(predictions_en) == len(references_en), (
    f"length mismatch: {len(predictions_en)} preds vs {len(references_en)} refs"
)
print(f"Loaded {len(predictions_en)} English prediction/reference pairs")

### 3b. English ROUGE

Multilingual ROUGE toolkit with `lang="english"` — uses English tokenization. Note this does not enable Porter stemming by default; to match ClidSum's reported numbers exactly, add `use_stemmer=True` to `RougeScorer(...)` inside `compute_rouge`.

In [ ]:
rouge_corpus_en, rouge_pairs_en = compute_rouge(
    predictions_en, references_en,
    rouge_types=ROUGE_TYPES, language=rouge_lang_en,
    use_stemmer=True
)

print("English ROUGE:")
print(f"  ROUGE-1 (R1) : {rouge_corpus_en['rouge1']:.2f}")
print(f"  ROUGE-2 (R2) : {rouge_corpus_en['rouge2']:.2f}")
print(f"  ROUGE-L (R-L): {rouge_corpus_en['rougeL']:.2f}")

### 3c. English BERTScore (raw)

Uses `roberta-large`, the BERTScore paper's recommended encoder for English. `num_layers=None` lets `bert_score` pick the paper-recommended layer automatically.

In [ ]:
bs_raw_corpus_en, bs_raw_pairs_en = compute_bertscore(
    predictions_en, references_en,
    model_type=bertscore_model_en,
    lang="en",
    num_layers=num_layers_en,
    rescale_with_baseline=False,
    batch_size=32,
)
print(f"English BERTScore F1 (raw): {bs_raw_corpus_en['f1']}")

### 3d. English BERTScore (rescaled with baseline)

Rescaling subtracts the BERTScore baseline (random-pair similarity) and renormalizes. The raw score for `roberta-large` sits compressed in the ~85–95 range; rescaled scores spread out over a wider range and are what the BERTScore paper recommends reporting.

In [ ]:
bs_rescaled_corpus_en, bs_rescaled_pairs_en = compute_bertscore(
    predictions_en, references_en,
    model_type=bertscore_model_en,
    lang="en",
    num_layers=num_layers_en,
    rescale_with_baseline=True,
    batch_size=32,
)
print(f"English BERTScore F1 (rescaled): {bs_rescaled_corpus_en['f1']}")

### 3e. Combine English results, inspect, and save

In [ ]:
corpus_scores_en = rouge_corpus_en | {
    "bs_f1_raw": bs_raw_corpus_en["f1"],
    "bs_f1_rescaled": bs_rescaled_corpus_en["f1"],
}
corpus_df_en = pd.DataFrame(corpus_scores_en, index=[0])
print(f"Corpus Eval Scores (English) of {DATASET_NAME}")
print(corpus_df_en)

In [ ]:
pair_cols_en = (
    {"reference": references_en, "prediction": predictions_en}
    | pd.DataFrame(rouge_pairs_en).to_dict("list")
    | {
        "bs_f1_raw": bs_raw_pairs_en,
        "bs_f1_rescaled": bs_rescaled_pairs_en,
    }
)
pair_results_df_en = pd.DataFrame(pair_cols_en)

print("── BERTScore F1 distribution (English) ──")
print(pair_results_df_en[["bs_f1_raw", "bs_f1_rescaled"]].describe().round(2))

print("\n── rougeL distribution (English) ──")
print(pair_results_df_en["rougeL"].describe().round(2))

In [ ]:
worst_en = pair_results_df_en.nsmallest(TOP_N_WORST, "rougeL")
print(f"── Top {TOP_N_WORST} worst English examples by ROUGE-L of {DATASET_NAME} ──")
worst_en

In [ ]:
corpus_df_en.to_csv(f"corpus_scores_en_{DATASET_NAME}.csv", index=False, encoding="utf-8-sig")
pair_results_df_en.to_csv(f"pair_scores_en_{DATASET_NAME}.csv", index=True, encoding="utf-8-sig")
print("Saved English scores to CSV.")